<a href="https://colab.research.google.com/github/Vdmtx/FBNeo-Android/blob/main/upscale_drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Upscale de Imagens com persistencia no Google Drive

**Como usar:**
1. Va em **Ambiente de execucao > Alterar tipo de ambiente de execucao > GPU T4**
2. **Primeira vez:** execute Celula 1 (setup — baixa tudo e salva no Drive)
3. **Proximas vezes:** pule a Celula 1, execute Celula 2 + Celula 3

**Modos de upscale:**
- Modo 3 (waifu2x) = textos, logos, tabelas, graficos
- Modo 1 (Real-ESRGAN x4plus) = fotografias


In [1]:
import os, glob, subprocess, sys, shutil
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_BASE    = '/content/drive/MyDrive/upscale_toolkit'
DRIVE_REPO    = f'{DRIVE_BASE}/Real-ESRGAN'
DRIVE_W2X     = f'{DRIVE_BASE}/waifu2x'
DRIVE_WEIGHTS = f'{DRIVE_REPO}/weights'

os.makedirs(DRIVE_BASE,    exist_ok=True)
os.makedirs(DRIVE_WEIGHTS, exist_ok=True)
os.makedirs(DRIVE_W2X,    exist_ok=True)

print('Pasta base no Drive:', DRIVE_BASE)

has_realesrgan = os.path.exists(f'{DRIVE_REPO}/inference_realesrgan.py')
has_w2x        = os.path.exists(f'{DRIVE_W2X}/waifu2x-ncnn-vulkan')

WEIGHTS_NEEDED = {
    'realesr-general-x4v3.pth':
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth',
    'RealESRGAN_x4plus.pth':
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    'RealESRGAN_x4plus_anime_6B.pth':
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth',
}
missing_weights = [n for n in WEIGHTS_NEEDED if not os.path.exists(f'{DRIVE_WEIGHTS}/{n}')]

print(f'Real-ESRGAN repo : {"OK" if has_realesrgan else "FALTANDO"}')
print(f'waifu2x binary   : {"OK" if has_w2x else "FALTANDO"}')
print(f'Pesos faltando   : {missing_weights if missing_weights else "nenhum"}')

if not has_realesrgan:
    print('Clonando Real-ESRGAN...')
    tmp = '/content/Real-ESRGAN-tmp'
    subprocess.run(['git','clone','https://github.com/xinntao/Real-ESRGAN.git', tmp,'-q'], check=True)
    shutil.copytree(tmp, DRIVE_REPO, dirs_exist_ok=True)
    shutil.rmtree(tmp)
    print('Repositorio salvo no Drive.')

if not has_w2x:
    print('Baixando waifu2x-ncnn-vulkan...')
    w2x_url = 'https://github.com/nihui/waifu2x-ncnn-vulkan/releases/download/20220728/waifu2x-ncnn-vulkan-20220728-ubuntu.zip'
    subprocess.run(['wget','-q','-O','/tmp/w2x.zip', w2x_url], check=True)
    subprocess.run(['unzip','-q','/tmp/w2x.zip','-d','/tmp/w2x'], check=True)
    extracted = glob.glob('/tmp/w2x/waifu2x-ncnn-vulkan-*')
    if extracted:
        for item in os.listdir(extracted[0]):
            src = os.path.join(extracted[0], item)
            dst = os.path.join(DRIVE_W2X, item)
            if os.path.isdir(src):
                shutil.copytree(src, dst, dirs_exist_ok=True)
            else:
                shutil.copy2(src, dst)
    print('waifu2x salvo no Drive.')

for fname, url in WEIGHTS_NEEDED.items():
    dest = f'{DRIVE_WEIGHTS}/{fname}'
    if not os.path.exists(dest):
        print(f'Baixando peso: {fname}...')
        subprocess.run(['wget','-q','--show-progress','-O', dest, url], check=True)

print('SETUP COMPLETO. Nas proximas sessoes, pule direto para a Celula 2.')


Mounted at /content/drive
Pasta base no Drive: /content/drive/MyDrive/upscale_toolkit
Real-ESRGAN repo : OK
waifu2x binary   : OK
Pesos faltando   : nenhum
SETUP COMPLETO. Nas proximas sessoes, pule direto para a Celula 2.


In [2]:
import os, glob, subprocess, sys, shutil
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_BASE    = '/content/drive/MyDrive/upscale_toolkit'
DRIVE_REPO    = f'{DRIVE_BASE}/Real-ESRGAN'
DRIVE_W2X     = f'{DRIVE_BASE}/waifu2x'
LOCAL_REPO    = '/content/Real-ESRGAN'
LOCAL_W2X_BIN = '/content/waifu2x-ncnn-vulkan'

if not os.path.exists(f'{LOCAL_REPO}/inference_realesrgan.py'):
    print('Copiando Real-ESRGAN do Drive...')
    shutil.copytree(DRIVE_REPO, LOCAL_REPO, dirs_exist_ok=True)
else:
    print('Real-ESRGAN ja em /content')

w2x_src = f'{DRIVE_W2X}/waifu2x-ncnn-vulkan'
if os.path.exists(w2x_src) and not os.path.exists(LOCAL_W2X_BIN):
    shutil.copy2(w2x_src, LOCAL_W2X_BIN)
    for mdir in ['models-cunet','models-upconv_7_anime_style_art_rgb','models-upconv_7_photo']:
        src = f'{DRIVE_W2X}/{mdir}'
        if os.path.exists(src):
            shutil.copytree(src, f'/content/{mdir}', dirs_exist_ok=True)
    os.chmod(LOCAL_W2X_BIN, 0o755)
    print('waifu2x copiado')

os.chdir(LOCAL_REPO)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt','-q'], check=True)
subprocess.run([sys.executable,'setup.py','develop','-q'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
os.chdir('/content')

for p in glob.glob('/usr/local/lib/python*/dist-packages/basicsr/data/degradations.py'):
    with open(p,'r') as f: src = f.read()
    patched = src.replace(
        'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
        'from torchvision.transforms.functional import rgb_to_grayscale'
    )
    if patched != src:
        with open(p,'w') as f: f.write(patched)

import torch
gpu_ok   = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_ok else 'SEM GPU'
print(f'GPU: {gpu_name}')
print(f'Real-ESRGAN: OK')
print(f'waifu2x: {"OK" if os.path.exists(LOCAL_W2X_BIN) else "nao encontrado"}')
print('Pronto! Execute a Celula 3 para fazer o upscale.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copiando Real-ESRGAN do Drive...
waifu2x copiado
GPU: Tesla T4
Real-ESRGAN: OK
waifu2x: OK
Pronto! Execute a Celula 3 para fazer o upscale.


In [3]:
import os, subprocess, sys, shutil, math
from PIL import Image
import cv2
from google.colab import files
from IPython.display import display, HTML

LOCAL_REPO    = '/content/Real-ESRGAN'
LOCAL_W2X_BIN = '/content/waifu2x-ncnn-vulkan'
DRIVE_OUT     = '/content/drive/MyDrive/upscale_toolkit/outputs'
os.makedirs(DRIVE_OUT, exist_ok=True)

print('Envie a imagem (JPG, PNG, TIFF):')
uploaded = files.upload()
if not uploaded:
    raise SystemExit('Nenhum arquivo enviado.')

fname = list(uploaded.keys())[0]
in_path = '/content/input.png'
with open(in_path,'wb') as f: f.write(uploaded[fname])
img_pil = Image.open(in_path).convert('RGB')
img_pil.save(in_path, 'PNG')
orig_w, orig_h = img_pil.size

print('Modos:')
print('  1 -> Real-ESRGAN x4plus    (fotos, alta fidelidade)')
print('  2 -> Real-ESRGAN general   (fotos, mais rapido)')
print('  3 -> waifu2x cunet         (texto, logos, arte, graficos)')
print('  4 -> Real-ESRGAN anime     (anime/ilustracao)')

modo     = input('Modo (1-4): ').strip()
alvo_px  = int(input('Largura-alvo em pixels (ex: 5000): ').strip())
alvo_dpi = int(input('DPI final (ex: 300, 600, 1200): ').strip())

print(f'Original : {orig_w} x {orig_h} px')
print(f'Alvo     : {alvo_px} px | {alvo_dpi} DPI')

cur_path = in_path
cur_w    = orig_w
cur_h    = orig_h

if modo == '3':
    escala   = min(int(math.ceil(alvo_px / orig_w)), 32)
    validas  = [s for s in [1,2,4,8,16,32] if s >= escala]
    escala   = validas[0] if validas else 32
    print(f'waifu2x x{escala} ({orig_w}px -> ~{orig_w*escala}px)...')
    out_w2x = '/content/out_w2x.png'
    ret = subprocess.run([
        LOCAL_W2X_BIN, '-i', in_path, '-o', out_w2x,
        '-n', '0', '-s', str(escala), '-m', 'models-cunet',
    ], capture_output=True, text=True)
    if not os.path.exists(out_w2x):
        print('waifu2x falhou:', ret.stderr[-500:])
        raise SystemExit('Tente outro modo.')
    cur_path = out_w2x
    img_out = Image.open(cur_path)
    cur_w, cur_h = img_out.size
    print(f'Saida waifu2x: {cur_w} x {cur_h} px')
else:
    MODEL_MAP = {
        '1': 'RealESRGAN_x4plus',
        '2': 'realesr-general-x4v3',
        '4': 'RealESRGAN_x4plus_anime_6B',
    }
    model  = MODEL_MAP.get(modo, 'realesr-general-x4v3')
    MAX_IT = 5
    for step in range(1, MAX_IT + 1):
        if cur_w >= alvo_px:
            print(f'Alvo atingido: {cur_w}px'); break
        print(f'Etapa {step}/{MAX_IT}: {model} ({cur_w}px -> ~{cur_w*4}px)...')
        out_dir = f'/content/out_{step}'
        os.makedirs(out_dir, exist_ok=True)
        tmp_in = f'/content/tmp_{step}.png'
        cv2.imwrite(tmp_in, cv2.imread(cur_path), [cv2.IMWRITE_PNG_COMPRESSION, 0])
        cmd = [
            sys.executable, f'{LOCAL_REPO}/inference_realesrgan.py',
            '-n', model, '-i', tmp_in, '-o', out_dir,
            '-s', '4', '-t', '512',
            '--model_path', f'{LOCAL_REPO}/weights/{model}.pth',
        ]
        ret = subprocess.run(cmd, capture_output=True, text=True)
        outs = [f for f in os.listdir(out_dir) if f.endswith('.png')]
        if not outs:
            print('Falha:', ret.stderr[-500:]); break
        cur_path = os.path.join(out_dir, outs[0])
        img_cv = cv2.imread(cur_path)
        cur_h, cur_w = img_cv.shape[:2]
        print(f'   -> {cur_w} x {cur_h} px')
    else:
        print(f'Limite de {MAX_IT} iteracoes. Largura: {cur_w}px')

img_final = Image.open(cur_path).convert('RGB')
fw, fh    = img_final.size
if fw > alvo_px:
    ratio     = alvo_px / fw
    new_h     = int(fh * ratio)
    img_final = img_final.resize((alvo_px, new_h), Image.LANCZOS)
    fw, fh    = img_final.size
    print(f'Redimensionado para {fw} x {fh} px (Lanczos)')

out_name  = os.path.splitext(fname)[0] + f'_{alvo_dpi}dpi_{fw}px.tiff'
out_local = f'/content/{out_name}'
out_drive = f'{DRIVE_OUT}/{out_name}'

img_final.save(out_local, dpi=(alvo_dpi, alvo_dpi), compression='tiff_lzw')
shutil.copy2(out_local, out_drive)

fsize_mb = os.path.getsize(out_local) / 1_048_576
print(f'Resolucao : {fw} x {fh} px')
print(f'DPI       : {alvo_dpi}')
print(f'Tamanho   : {fsize_mb:.1f} MB')
print(f'Salvo em  : MyDrive/upscale_toolkit/outputs/{out_name}')

files.download(out_local)


Envie a imagem (JPG, PNG, TIFF):


Saving FIL_5410.JPG to FIL_5410.JPG
Modos:
  1 -> Real-ESRGAN x4plus    (fotos, alta fidelidade)
  2 -> Real-ESRGAN general   (fotos, mais rapido)
  3 -> waifu2x cunet         (texto, logos, arte, graficos)
  4 -> Real-ESRGAN anime     (anime/ilustracao)
Modo (1-4): 1
Largura-alvo em pixels (ex: 5000): 3000
DPI final (ex: 300, 600, 1200): 1200
Original : 4496 x 3000 px
Alvo     : 3000 px | 1200 DPI
Alvo atingido: 4496px
Redimensionado para 3000 x 2001 px (Lanczos)
Resolucao : 3000 x 2001 px
DPI       : 1200
Tamanho   : 17.0 MB
Salvo em  : MyDrive/upscale_toolkit/outputs/FIL_5410_1200dpi_3000px.tiff


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>